# Colab: `log.py` Experiments

Use this notebook to run `log.py` with different parameter settings and compare metrics.

It supports:
- single experiment runs
- `--compare-all` runs
- a dedicated `tabular_only` hidden-dim study with a paper-style summary table

In [ ]:
import sys
assert 'google.colab' in sys.modules, 'Please run this notebook in Google Colab.'
print('Running in Colab')

In [ ]:
%cd /content/
!rm -rf project_viral
!git clone https://github.com/Zmjjkk880/project_viral
%cd /content/project_viral
!ls

In [ ]:
!pip -q install sentence-transformers scikit-learn pandas numpy torch

In [ ]:
# Optional: generate processed CSV if you do not already have it.
# If your GitHub repo already contains the target CSV, you can skip this cell.
!python preprocess.py --views-threshold 20000 --engagement-threshold 0.06

In [ ]:
import importlib
import re
import shlex
import subprocess
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

import log as log_module


log_module = importlib.reload(log_module)
REPO_DIR = Path('/content/project_viral')


def build_log_command(config: dict):
    cmd = ['python', 'log.py']

    scalar_args = {
        'data-path': config.get('data_path'),
        'batch-size': config.get('batch_size'),
        'epochs': config.get('epochs'),
        'learning-rate': config.get('learning_rate'),
        'hidden-dim': config.get('hidden_dim'),
        'dropout': config.get('dropout'),
        'test-size': config.get('test_size'),
        'threshold': config.get('threshold'),
        'token-mixer': config.get('token_mixer'),
        'proj-dim': config.get('proj_dim'),
        'output-csv': config.get('output_csv'),
    }

    for key, value in scalar_args.items():
        if value is not None:
            cmd.extend([f'--{key}', str(value)])

    if config.get('compare_all', False):
        cmd.append('--compare-all')

    return cmd


def parse_single_summary(output: str):
    metrics = {}
    patterns = {
        'token_mixer': r'Token mixer: (.+)',
        'test_loss': r'Test Loss: ([0-9.]+)',
        'test_auc': r'Test ROC-AUC: ([0-9.]+)',
        'accuracy': r'Accuracy: ([0-9.]+)',
        'macro_f1': r'Macro F1: ([0-9.]+)',
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, output)
        if match:
            metrics[key] = match.group(1)
    return metrics


def parse_compare_all_summary(output: str):
    rows = []
    pattern = re.compile(
        r'^(text_only|tabular_only|raw_concat|projected_concat|weighted_sum|attention_pool): '
        r'loss=([0-9.]+), auc=([0-9.]+), accuracy=([0-9.]+), macro_f1=([0-9.]+)$',
        re.MULTILINE,
    )
    for token_mixer, loss, auc, accuracy, macro_f1 in pattern.findall(output):
        rows.append({
            'token_mixer': token_mixer,
            'test_loss': float(loss),
            'test_auc': float(auc),
            'accuracy': float(accuracy),
            'macro_f1': float(macro_f1),
        })
    return rows


def run_log_experiment(config: dict, capture_output: bool = True):
    cmd = build_log_command(config)
    print('Running command:')
    print(' '.join(shlex.quote(part) for part in cmd))

    completed = subprocess.run(
        cmd,
        cwd=REPO_DIR,
        text=True,
        capture_output=capture_output,
    )

    if capture_output:
        print(completed.stdout)
        if completed.stderr:
            print('STDERR:')
            print(completed.stderr)

    completed.check_returncode()

    output = completed.stdout if capture_output else ''
    if config.get('compare_all', False):
        return {
            'command': cmd,
            'raw_output': output,
            'rows': parse_compare_all_summary(output),
        }

    return {
        'command': cmd,
        'raw_output': output,
        'summary': parse_single_summary(output),
    }


def prepare_shared_log_data(config: dict):
    log_module.set_seed(log_module.RANDOM_STATE)
    df = log_module.load_data(config['data_path'])
    train_df, test_df = log_module.split_data(df, config['test_size'])
    x_train, x_test, y_train, y_test = log_module.build_features(train_df, test_df)

    text_dim = x_train.shape[1] - len(log_module.NUMERIC_COLUMNS) - len(
        pd.get_dummies(train_df[log_module.CATEGORICAL_COLUMNS], drop_first=False).columns
    )
    tab_dim = x_train.shape[1] - text_dim

    return {
        'train_df': train_df,
        'test_df': test_df,
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'text_dim': text_dim,
        'tab_dim': tab_dim,
    }


def find_best_threshold(labels, probs, start: float = 0.05, stop: float = 0.95, step: float = 0.01):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs)
    thresholds = np.round(np.arange(start, stop + step / 2, step), 2)

    best_row = None
    for threshold in thresholds:
        preds = (probs >= threshold).astype(int)
        row = {
            'threshold': float(threshold),
            'accuracy': float(accuracy_score(labels, preds)),
            'macro_f1': float(f1_score(labels, preds, average='macro')),
        }

        if best_row is None:
            best_row = row
            continue

        better_macro_f1 = row['macro_f1'] > best_row['macro_f1'] + 1e-12
        same_macro_f1 = np.isclose(row['macro_f1'], best_row['macro_f1'])
        closer_to_fixed = abs(row['threshold'] - 0.5) < abs(best_row['threshold'] - 0.5)
        same_distance = np.isclose(abs(row['threshold'] - 0.5), abs(best_row['threshold'] - 0.5))
        lower_threshold = row['threshold'] < best_row['threshold']

        if better_macro_f1 or (same_macro_f1 and (closer_to_fixed or (same_distance and lower_threshold))):
            best_row = row

    return best_row


def run_inprocess_single(config: dict, shared_data: dict = None):
    if shared_data is None:
        shared_data = prepare_shared_log_data(config)

    log_module.set_seed(log_module.RANDOM_STATE)
    args = SimpleNamespace(**config)
    result = log_module.train_single_model(
        token_mixer=config['token_mixer'],
        x_train=shared_data['x_train'],
        x_test=shared_data['x_test'],
        y_train=shared_data['y_train'],
        y_test=shared_data['y_test'],
        text_dim=shared_data['text_dim'],
        tab_dim=shared_data['tab_dim'],
        args=args,
    )

    best_metrics = find_best_threshold(result['labels'], result['probs'])
    return {
        'name': config.get('name', config['token_mixer']),
        'hidden_dim': config['hidden_dim'],
        'token_mixer': config['token_mixer'],
        'proj_dim': config['proj_dim'],
        'test_loss': float(result['test_loss']),
        'test_auc': float(result['test_auc']),
        'fixed_accuracy': float(result['test_accuracy']),
        'fixed_macro_f1': float(result['test_macro_f1']),
        'best_threshold': float(best_metrics['threshold']),
        'best_macro_f1': float(best_metrics['macro_f1']),
    }


def style_experiment_table(df: pd.DataFrame, caption: str):
    return (
        df.style.format(
            {
                'test_loss': '{:.6f}',
                'test_auc': '{:.6f}',
                'fixed_accuracy': '{:.6f}',
                'fixed_macro_f1': '{:.6f}',
                'best_threshold': '{:.6f}',
                'best_macro_f1': '{:.6f}',
            }
        ).set_caption(caption)
    )


In [ ]:
# Single-run config
single_config = {
    'data_path': '/content/project_viral/data/processed/processed_data_20000_0.06.csv',
    'batch_size': 64,
    'epochs': 50,
    'learning_rate': 1e-5,
    'hidden_dim': 128,
    'dropout': 0.3,
    'test_size': 0.2,
    'threshold': 0.5,
    'token_mixer': 'raw_concat',
    'proj_dim': 64,
    'compare_all': False,
    'output_csv': '/content/project_viral/comparison_outputs/model_probabilities.csv',
}

In [ ]:
single_result = run_log_experiment(single_config)
pd.DataFrame([single_result['summary']])

In [ ]:
# Compare-all config
compare_all_config = {
    'data_path': '/content/project_viral/data/processed/processed_data_20000_0.06.csv',
    'batch_size': 64,
    'epochs': 50,
    'learning_rate': 1e-5,
    'hidden_dim': 128,
    'dropout': 0.3,
    'test_size': 0.2,
    'threshold': 0.5,
    'token_mixer': 'raw_concat',
    'proj_dim': 64,
    'compare_all': True,
    'output_csv': '/content/project_viral/comparison_outputs/model_probabilities.csv',
}

In [ ]:
compare_all_result = run_log_experiment(compare_all_config)
pd.DataFrame(compare_all_result['rows']).sort_values(['test_auc', 'macro_f1'], ascending=False)

In [ ]:
# Hidden-dim study: tabular_only only, while keeping batch_size / epochs / learning_rate fixed.
# For the later proj_dim study, keep the same structure and only swap the sweep target.
hidden_dim_base_config = {
    'data_path': '/content/project_viral/data/processed/processed_data_20000_0.06.csv',
    'batch_size': 64,
    'epochs': 50,
    'learning_rate': 1e-5,
    'hidden_dim': 128,
    'dropout': 0.3,
    'test_size': 0.2,
    'threshold': 0.5,
    'token_mixer': 'tabular_only',
    'proj_dim': 64,
    'compare_all': False,
}

hidden_dim_values = [128, 64, 256]
shared_hidden_dim_data = prepare_shared_log_data(hidden_dim_base_config)

hidden_dim_configs = [
    {
        **hidden_dim_base_config,
        'name': f'tabular_only_hidden_{hidden_dim}',
        'hidden_dim': hidden_dim,
    }
    for hidden_dim in hidden_dim_values
]

In [ ]:
hidden_dim_rows = []
for index, config in enumerate(hidden_dim_configs):
    print('\n' + '=' * 100)
    print(
        f"Experiment {index}: token_mixer={config['token_mixer']}, "
        f"hidden_dim={config['hidden_dim']}, proj_dim={config['proj_dim']}"
    )
    hidden_dim_rows.append(run_inprocess_single(config, shared_hidden_dim_data))

hidden_dim_df = pd.DataFrame(hidden_dim_rows)[
    [
        'hidden_dim',
        'token_mixer',
        'proj_dim',
        'test_loss',
        'test_auc',
        'fixed_accuracy',
        'fixed_macro_f1',
        'best_threshold',
        'best_macro_f1',
    ]
]
hidden_dim_df.index.name = 'index'

output_path = REPO_DIR / 'comparison_outputs' / 'tabular_only_hidden_dim_results.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
hidden_dim_df.to_csv(output_path)

print(f'Saved hidden-dim table to: {output_path}')
display(
    style_experiment_table(
        hidden_dim_df.reset_index(),
        'Table 4.x The result of different hidden_dim settings under tabular_only'
    )
)

hidden_dim_df.reset_index()